# Data Preprocessing

## Import Dependencies

In [7]:
import os
import warnings

import pandas as pd

from mappers import to_numeric, encode, get_keys

warnings.filterwarnings("ignore")

## Data Loading

In [8]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
path = os.path.join(root, "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

## Data Cleaning

#### Remove unnecessary features
1. `Category URL`

2. `Service URL`

3. `Offer URL`

4. `Offer Name`

5. `Owner URL`

6. `Owner Name`


In [9]:
unnecessary_features = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]

dataset.drop(columns=unnecessary_features, inplace=True)

#### Convert text-based values to numeric values
1. Time: `Duration`, `Offer Response Time`, and `Owner Response Time`.

2. Percentage: `Owner Completion Rate`.

3. Boolean: `Owner Verified`.

4. Money: `Price`.


In [10]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")

dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)

dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)

dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time", "Owner Level"])

## Encoding Categorical Features

In [5]:
category_encoded = pd.DataFrame(dataset["Category Name"].apply(lambda value: encode(value, "Category Name")).tolist(), columns=[key for key in get_keys("Category Name")])

service_encoded = pd.DataFrame(dataset["Service Name"].apply(lambda value: encode(value, "Service Name")).tolist(), columns=[key for key in get_keys("Service Name")])

dataset = dataset.drop(columns=["Category Name", "Service Name"])

dataset = pd.concat([dataset, category_encoded, service_encoded], axis=1)

## Handling Missing Values

In [ ]:
# TODO: use KNN

## Save the cleaned classification dataset

In [32]:
path = os.path.join(root, "clean.csv")

dataset.to_csv(path_or_buf=path, index=False)